# DynamoDB → S3 → Glue daily

Notebook này là **control panel** của pipeline. Logic AWS nằm trong `deploy.py`, `orchestrator.py` và `glue_job.py` để dễ test và deploy; notebook giúp chạy từng bước và quan sát kết quả.

Flow: EventBridge → Step Functions → DynamoDB Export → S3 raw → Glue → Hive-partitioned Parquet.

## 1. Chuẩn bị

Copy `.env.example` thành `.env`, điền bucket/table thật. Chạy cell cài dependency một lần cho đúng kernel.

In [ ]:
%pip install boto3 python-dotenv -q

In [ ]:
import importlib
import sys
from pathlib import Path

notebook_dir = Path.cwd()
if not (notebook_dir / 'deploy.py').is_file():
    notebook_dir = Path.cwd() / 'dynamodb_pipeline'
sys.path.insert(0, str(notebook_dir))
sys.modules.pop('deploy', None)  # Tránh import nhầm deploy.py của pipeline khác.
import deploy

importlib.reload(deploy)
cfg = deploy.config()
print(f'Region  : {cfg.region}')
print(f'Table   : {cfg.table}')
print(f'Raw     : s3://{cfg.bucket}/{cfg.raw_prefix}/')
print(f'Curated : s3://{cfg.bucket}/{cfg.curated_prefix}/year=YYYY/month=MM/day=DD/')
print(f'Schedule: {cfg.schedule}')

## 2. Deploy/update hạ tầng

`setup()` idempotent: chạy lại sẽ update resource cùng tên. Nó tạo IAM roles, Lambda, control table, Glue job/catalog, Step Functions và EventBridge rule.

In [ ]:
# deploy.setup()  # Bỏ dấu # khi đã kiểm tra config ở cell trên.

## 3. Chạy thử thủ công

Lần đầu là full export; các lần sau là incremental export theo watermark. Lưu ARN để kiểm tra trạng thái.

In [ ]:
# execution_arn = deploy.run_once()

In [ ]:
# deploy.execution_status(execution_arn)

## 4. Kiểm tra kết quả

Execution cần `SUCCEEDED`. Curated phải có `year=.../month=.../day=...`. Raw DynamoDB vẫn dùng layout AWS-managed `AWSDynamoDB/<export-id>/data/`; đây là hành vi native export, không phải Hive curated layout.

## 5. Destroy pipeline

Xóa orchestration/compute do pipeline tạo. **Giữ nguyên DynamoDB source table, S3 bucket và mọi object raw/curated.**

In [ ]:
# deploy.destroy()  # Bỏ dấu # chỉ khi thật sự muốn teardown.